# Time Series Sales Forecasting System

## Task 2 — Business Context

Forecasting future sales lets an e-commerce business plan inventory, staffing, marketing
spend and cash flow ahead of time, instead of reacting after demand has already shifted.
This notebook builds a complete time-series forecasting pipeline: cleaning and organizing
date-indexed sales data, exploring trend/seasonality/anomalies, engineering time-aware
features, comparing multiple forecasting approaches with a **chronological** (never
shuffled) train-test split, evaluating with MAE/RMSE/MAPE, producing a future forecast, and
summarizing what the patterns imply for the business.

### Dataset

**No real historical sales export was available for this task**, so this notebook
**generates a realistic synthetic daily sales dataset** (~3 years) directly in Section 2,
with a deliberately embedded trend, weekly seasonality, yearly (holiday) seasonality,
promotional spikes, and a few anomalies/missing days — the same kinds of patterns a real
e-commerce sales series would show. Every later section (EDA, feature engineering,
modeling, evaluation) is written generically against a `date` + `sales` dataframe, so it
works unchanged if a real sales export (e.g. daily order totals from the store's database)
is substituted in Section 2 later — only the data-generation cell would need to be replaced
with a `pd.read_csv(...)` call.

This notebook is **self-contained**: everything is defined in the cells below, nothing is
imported from an external module, and nothing is written to disk.

> **Note:** Written to be run top-to-bottom (`Run All`) later; it has not been executed yet
> — no outputs/plots are attached, only the pipeline code and the reasoning behind each step.


## 1. Environment Setup

- `pandas` / `numpy` — date-indexed data handling
- `matplotlib` / `seaborn` — visualization
- `statsmodels` — classical decomposition (trend/seasonality/residual) and SARIMA
- `scikit-learn` — regression-based forecasting models and evaluation metrics
- `xgboost` / `lightgbm` — gradient-boosted regressors, strong for feature-based forecasting

A fixed `RANDOM_STATE` makes the synthetic data and any stochastic models reproducible.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

print("Setup complete.")

## 2. Generating an Assumed Sales Dataset

Since a real historical sales export was not available for this task, we synthesize a
daily e-commerce sales series that mirrors the structure real retail data typically has, so
every later technique (decomposition, lag/rolling features, forecasting models) has
something realistic to work against:

- **~3 years of daily data** (2023-01-01 through 2025-12-31).
- **Upward trend** — the business is assumed to be growing steadily.
- **Weekly seasonality** — higher sales on weekends, as is typical for consumer e-commerce.
- **Yearly seasonality** — a broad November-December holiday-shopping peak plus a smaller
  mid-year sale bump.
- **Promotional spikes** — occasional one-day flash-sale spikes at random dates.
- **Random noise** — day-to-day demand variation.
- **Missing days** — a handful of dates with no recorded sales (e.g. a reporting outage),
  to exercise the cleaning step.
- **Outlier anomalies** — a few implausible one-off spikes/drops to exercise anomaly
  detection.

This keeps the rest of the notebook honest about being demonstrated on **assumed**, not
real, data — swap this cell for a real `pd.read_csv("sales.csv")` load when real historical
sales become available; nothing downstream needs to change as long as the result is a
dataframe with a `date` column and a `sales` column.


In [ ]:
date_range = pd.date_range(start="2023-01-01", end="2025-12-31", freq="D")
n = len(date_range)
t = np.arange(n)

# Trend: steady growth from a baseline
trend = 800 + t * 0.45

# Weekly seasonality: weekend uplift (Sat=5, Sun=6 in .dayofweek)
dow = date_range.dayofweek
weekly_seasonality = np.where(dow >= 5, 220, 0) + np.where(dow == 4, 90, 0)  # Fri smaller bump

# Yearly seasonality: holiday peak around Nov-Dec, smaller mid-year sale bump in July
day_of_year = date_range.dayofyear
holiday_peak = 500 * np.exp(-((day_of_year - 335) ** 2) / (2 * 20 ** 2))       # ~Dec 1
midyear_bump = 200 * np.exp(-((day_of_year - 195) ** 2) / (2 * 10 ** 2))       # ~mid July
yearly_seasonality = holiday_peak + midyear_bump

# Random promotional flash-sale spikes (sparse)
rng = np.random.default_rng(RANDOM_STATE)
promo_days = rng.choice(n, size=25, replace=False)
promo_spikes = np.zeros(n)
promo_spikes[promo_days] = rng.uniform(300, 700, size=len(promo_days))

# Random noise
noise = rng.normal(0, 60, size=n)

sales = trend + weekly_seasonality + yearly_seasonality + promo_spikes + noise
sales = np.clip(sales, a_min=50, a_max=None)  # sales can't go negative

df = pd.DataFrame({"date": date_range, "sales": sales})

# Inject a few missing days (simulating a reporting gap)
missing_idx = rng.choice(n, size=8, replace=False)
df.loc[missing_idx, "sales"] = np.nan

# Inject a few anomalous outliers (data-entry errors / one-off events)
anomaly_idx = rng.choice(n, size=5, replace=False)
df.loc[anomaly_idx, "sales"] = df.loc[anomaly_idx, "sales"] * rng.choice([0.15, 4.0], size=5)

print(f"Generated {len(df)} daily records from {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 3. Time-Series Data Cleaning & Date Organization

Time-series data needs its own cleaning checklist beyond a standard tabular dataset:

1. **Parse and sort by date** — forecasting logic (lags, rolling windows, chronological
   splits) is meaningless if rows are not in strict date order.
2. **Enforce a complete, gap-free date index** — reindex to every calendar day in range so
   missing days become explicit `NaN`s rather than silently shifting later date-based
   features (e.g. a lag-7 feature must skip exactly 7 *calendar* days, not 7 *rows*).
3. **Impute missing values with time-aware interpolation** — a straight median fill would
   erase the trend/seasonality right where data is missing; **linear interpolation** (or
   seasonal interpolation) preserves the local trajectory instead.
4. **Detect and treat anomalies** — flag points that deviate sharply from a rolling
   median/IQR band, since a handful of extreme data-entry errors can distort scaling and
   destabilize evaluation metrics.
5. **Extract calendar features** — year, month, day, day-of-week, week-of-year, is-weekend,
   is-month-start/end — the raw building blocks for both EDA and later feature engineering.


In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

full_index = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
df = df.set_index("date").reindex(full_index)
df.index.name = "date"

print(f"Missing values after reindexing to a complete calendar: {df['sales'].isnull().sum()}")
df.head()

In [ ]:
df["sales"] = df["sales"].interpolate(method="linear")
print(f"Missing values after interpolation: {df['sales'].isnull().sum()}")

In [ ]:
# Anomaly detection: flag points far outside a rolling median +/- N*MAD band
window = 14
rolling_median = df["sales"].rolling(window, center=True, min_periods=5).median()
rolling_mad = (df["sales"] - rolling_median).abs().rolling(window, center=True, min_periods=5).median()

threshold = 5  # modified z-score-style threshold on MAD
modified_z = 0.6745 * (df["sales"] - rolling_median) / rolling_mad.replace(0, np.nan)
df["is_anomaly"] = (modified_z.abs() > threshold).fillna(False)

print(f"Flagged anomalies: {df['is_anomaly'].sum()}")
df[df["is_anomaly"]]

In [ ]:
# Treat anomalies by replacing with the local rolling median (winsorizing point outliers),
# rather than dropping the date - forecasting needs a value for every calendar day.
df.loc[df["is_anomaly"], "sales"] = rolling_median[df["is_anomaly"]]

print("Anomalies treated (replaced with local rolling median).")
df["sales"].describe()

In [ ]:
# Calendar feature extraction
df["year"] = df.index.year
df["month"] = df.index.month
df["day"] = df.index.day
df["day_of_week"] = df.index.dayofweek
df["day_name"] = df.index.day_name()
df["week_of_year"] = df.index.isocalendar().week.astype(int)
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["is_month_start"] = df.index.is_month_start.astype(int)
df["is_month_end"] = df.index.is_month_end.astype(int)
df["quarter"] = df.index.quarter

df.head()

## 4. Exploratory Analysis — Trend, Seasonality & Patterns

### 4.1 Overall Sales Trend

We start with the raw series to visually confirm the presence of a trend and get a first
look at volatility and any remaining irregularities.


In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(df.index, df["sales"], linewidth=0.8, color="#4C72B0")
plt.plot(df.index, df["sales"].rolling(30).mean(), color="#DD8452", linewidth=2, label="30-day rolling mean")
plt.title("Daily Sales Over Time (with 30-day Rolling Mean)")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.show()

### 4.2 Seasonal Decomposition

Classical decomposition splits the series into **trend**, **seasonal**, and **residual**
components, making it easy to confirm (rather than just eyeball) that a repeating weekly
pattern and a long-run growth trend both exist, and to see what is left over (the residual)
once both are accounted for.


In [ ]:
decomposition = seasonal_decompose(df["sales"], model="additive", period=7)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
decomposition.observed.plot(ax=axes[0], title="Observed")
decomposition.trend.plot(ax=axes[1], title="Trend")
decomposition.seasonal.plot(ax=axes[2], title="Seasonal (weekly)")
decomposition.resid.plot(ax=axes[3], title="Residual")
plt.tight_layout()
plt.show()

### 4.3 Day-of-Week & Monthly Seasonality

Beyond the additive decomposition (fixed at a 7-day period), we look directly at average
sales by day-of-week and by month/quarter to characterize *which* days and *which* months
drive the seasonal pattern — this is the kind of concrete detail a business can act on
directly (e.g. staffing up on weekends, planning inventory ahead of a Q4 peak).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

dow_avg = df.groupby("day_name")["sales"].mean().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)
sns.barplot(x=dow_avg.index, y=dow_avg.values, ax=axes[0], hue=dow_avg.index, legend=False)
axes[0].set_title("Average Sales by Day of Week")
axes[0].tick_params(axis="x", rotation=45)

month_avg = df.groupby("month")["sales"].mean()
sns.barplot(x=month_avg.index, y=month_avg.values, ax=axes[1], hue=month_avg.index, legend=False)
axes[1].set_title("Average Sales by Month")
axes[1].set_xlabel("Month")

plt.tight_layout()
plt.show()

In [ ]:
# Year-over-year comparison, to distinguish "trend" from "seasonality" visually
plt.figure(figsize=(14, 6))
for yr, group in df.groupby("year"):
    plt.plot(group["day_of_week"].index.dayofyear, group["sales"].rolling(7).mean(), label=str(yr))
plt.title("Year-over-Year Sales Comparison (7-day Rolling Mean, by Day-of-Year)")
plt.xlabel("Day of Year")
plt.ylabel("Sales (7-day rolling mean)")
plt.legend(title="Year")
plt.tight_layout()
plt.show()

### 4.4 Stationarity Check

Several classical forecasting models (like ARIMA/SARIMA) assume a stationary series (constant
mean/variance over time). Since we already know this series has trend and seasonality, we
expect the **Augmented Dickey-Fuller (ADF) test** to reject stationarity on the raw series —
confirming why we difference the series (or let SARIMA's `d`/`D` orders handle it) before
fitting an ARIMA-family model.


In [ ]:
adf_result = adfuller(df["sales"].dropna())
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")
print("Stationary" if adf_result[1] < 0.05 else "Non-stationary (trend/seasonality present, as expected)")

# First-order differencing typically removes trend; compare ADF after differencing
adf_diff = adfuller(df["sales"].diff().dropna())
print(f"\nAfter first-order differencing:")
print(f"ADF Statistic: {adf_diff[0]:.4f}")
print(f"p-value: {adf_diff[1]:.4f}")

## 5. Feature Engineering — Time-Based, Lag & Rolling-Window Features

Forecasting models that aren't pure time-series models (e.g. tree ensembles) can't "see"
time directly — they need the temporal structure translated into explicit numeric
features. We build three families:

1. **Time-based / calendar features** (already partly built in Section 3): cyclical
   encodings of day-of-week and month so a model can learn "Sunday is close to Monday" and
   "December is close to January" instead of treating them as unrelated integers.
2. **Lag features** — the sales value N days ago (`lag_1`, `lag_7`, `lag_14`, `lag_28`) —
   the single strongest predictor of "sales tomorrow" is usually "sales recently", and the
   7/14/28-day lags specifically capture weekly seasonality echoes.
3. **Rolling-window features** — rolling mean/std/min/max over trailing 7/14/30-day windows
   — smooth out day-to-day noise and capture the current momentum/volatility level.

**Critical leakage rule:** every lag/rolling feature is computed using only *past* values
relative to each row (`.shift()` before `.rolling()`), so no feature for date *t* ever uses
information from date *t* or later — otherwise the model would effectively "see the future"
during training and its offline evaluation would be meaninglessly optimistic.


In [ ]:
# Cyclical encoding of periodic calendar features, so e.g. Dec (12) and Jan (1) are
# numerically close instead of maximally far apart.
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# Lag features (value shifted forward so it reflects a strictly past observation)
for lag in [1, 7, 14, 28]:
    df[f"lag_{lag}"] = df["sales"].shift(lag)

# Rolling-window statistics, computed on the *shifted* series so the window for date t
# never includes date t itself.
shifted_sales = df["sales"].shift(1)
for window in [7, 14, 30]:
    df[f"rolling_mean_{window}"] = shifted_sales.rolling(window).mean()
    df[f"rolling_std_{window}"] = shifted_sales.rolling(window).std()
    df[f"rolling_min_{window}"] = shifted_sales.rolling(window).min()
    df[f"rolling_max_{window}"] = shifted_sales.rolling(window).max()

# Trend index: a simple linear time counter, lets tree models capture long-run growth
df["time_index"] = np.arange(len(df))

print(f"Shape after feature engineering: {df.shape}")
feature_cols = [c for c in df.columns if c not in
                ["sales", "is_anomaly", "day_name"]]
print(f"Candidate feature columns ({len(feature_cols)}): {feature_cols}")

# Drop the warm-up rows where the longest lag/rolling window isn't yet available
df_model = df.dropna(subset=feature_cols + ["sales"]).copy()
print(f"\nRows dropped for lag/rolling warm-up: {len(df) - len(df_model)}")
print(f"Model-ready shape: {df_model.shape}")

## 6. Chronological Train-Test Split

**This is the one rule that is non-negotiable for time series: never shuffle, never use a
random split.** A random split would let the model train on data *after* the test period
and "predict the past" using future information — wildly overstating real-world accuracy.
Instead we hold out the **most recent block of time** as the test set, mimicking exactly the
real deployment scenario (train on history, forecast forward).

We reserve the **last 90 days** as the test set and everything before that as training.


In [ ]:
test_horizon_days = 90
split_date = df_model.index.max() - pd.Timedelta(days=test_horizon_days - 1)

train = df_model[df_model.index < split_date]
test = df_model[df_model.index >= split_date]

feature_cols = [c for c in df_model.columns if c not in ["sales", "is_anomaly", "day_name"]]

X_train, y_train = train[feature_cols], train["sales"]
X_test, y_test = test[feature_cols], test["sales"]

print(f"Train period: {train.index.min().date()} to {train.index.max().date()} ({len(train)} days)")
print(f"Test period:  {test.index.min().date()} to {test.index.max().date()} ({len(test)} days)")

## 7. Building & Comparing Forecasting Models

We compare two families of approaches, since neither is universally best:

**Classical statistical models** — model the series' own autocorrelation/seasonality
structure directly:
- **Holt-Winters Exponential Smoothing** — explicitly models level, trend and (additive)
  seasonality; fast, robust, a strong classical baseline for a series like this one.
- **SARIMAX** — seasonal ARIMA; captures autocorrelation and a weekly seasonal period (7)
  explicitly through its `(P, D, Q, s)` seasonal order.

**Feature-based machine learning models** — treat forecasting as supervised regression on
the engineered lag/rolling/calendar features from Section 5:
- **Linear Regression** — interpretable baseline; can a straight-line combination of the
  engineered features explain most of the signal?
- **Random Forest Regressor** — non-linear, handles feature interactions without scaling.
- **Gradient Boosting Regressor** (sklearn), **XGBoost**, **LightGBM** — usually the
  strongest performers on feature-rich tabular/time-series-as-regression problems.

All ML models are trained on the **same chronological split** from Section 6.


In [ ]:
results = []
predictions = {}

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    results.append({"model": name, "MAE": mae, "RMSE": rmse, "MAPE (%)": mape})
    predictions[name] = y_pred
    return mae, rmse, mape

### 7.1 Holt-Winters Exponential Smoothing


In [ ]:
hw_model = ExponentialSmoothing(
    train["sales"], trend="add", seasonal="add", seasonal_periods=7
).fit()

hw_forecast = hw_model.forecast(len(test))
hw_forecast.index = test.index
evaluate("Holt-Winters", y_test, hw_forecast)
print("Holt-Winters fitted.")

### 7.2 SARIMAX

Order `(1, 1, 1)` with a weekly seasonal order `(1, 1, 1, 7)` is a reasonable general
starting point given the ADF test showed the raw series is non-stationary but first
differencing helps, and the decomposition confirmed a 7-day seasonal cycle. In practice
these orders would be tuned further (e.g. via `pmdarima.auto_arima` or a grid search over
AIC), which is noted as a follow-up rather than performed exhaustively here to keep runtime
reasonable.


In [ ]:
sarimax_model = SARIMAX(
    train["sales"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

sarimax_forecast = sarimax_model.forecast(steps=len(test))
sarimax_forecast.index = test.index
evaluate("SARIMAX", y_test, sarimax_forecast)
print("SARIMAX fitted.")

### 7.3 Feature-Based Machine Learning Models


In [ ]:
ml_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

for name, model in ml_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    evaluate(name, y_test, y_pred)
    print(f"{name} fitted.")

## 8. Model Comparison — MAE, RMSE, MAPE

- **MAE (Mean Absolute Error)** — average absolute miss, in the same units as sales; easy to
  communicate to the business ("on average we're off by $X/day").
- **RMSE (Root Mean Squared Error)** — penalizes large misses more heavily than MAE; useful
  for catching models that are usually fine but occasionally very wrong.
- **MAPE (Mean Absolute Percentage Error)** — scale-free relative error, useful for
  comparing forecast quality across different sales *levels* (e.g. low season vs. peak
  season), though it can be unstable if actual sales are ever near zero (not a concern
  here since sales stay comfortably positive).

We report all three since each highlights a different failure mode; the "best" model is the
one that is consistently strong across all three, not just the top of any single metric.


In [ ]:
comparison_df = pd.DataFrame(results).set_index("model").sort_values("RMSE")
comparison_df.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ["MAE", "RMSE", "MAPE (%)"]):
    sns.barplot(x=comparison_df.index, y=comparison_df[metric], ax=ax, hue=comparison_df.index, legend=False)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = comparison_df["RMSE"].idxmin()
print(f"Best model on the held-out test period (by RMSE): {best_model_name}")

## 9. Actual vs. Predicted — Test Period


In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(train.index[-60:], train["sales"].iloc[-60:], label="Train (last 60 days)", color="gray")
plt.plot(test.index, y_test, label="Actual", color="black", linewidth=2)

palette = sns.color_palette("husl", len(predictions))
for (name, y_pred), color in zip(predictions.items(), palette):
    plt.plot(test.index, y_pred, label=name, alpha=0.8, color=color)

plt.axvline(test.index.min(), linestyle="--", color="red", alpha=0.5, label="Train/Test Split")
plt.title("Actual vs. Predicted Sales — Test Period")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend(loc="upper left", ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Focused view: best model only, easier to judge fit quality
plt.figure(figsize=(16, 5))
plt.plot(test.index, y_test, label="Actual", color="black", linewidth=2)
plt.plot(test.index, predictions[best_model_name], label=f"Predicted ({best_model_name})", color="#DD8452")
plt.fill_between(test.index, y_test, predictions[best_model_name], color="gray", alpha=0.2)
plt.title(f"Best Model — Actual vs. Predicted ({best_model_name})")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis for the best model: are errors random, or is there a pattern
# the model is systematically missing (e.g. still under-predicting weekends)?
residuals = y_test - predictions[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(test.index, residuals, marker="o", markersize=3, linewidth=0.8)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals Over Time")

sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title("Residual Distribution")
plt.tight_layout()
plt.show()

## 10. Forecasting Future Sales

With the best model selected and validated on the held-out period, we refit it on the
**full available history** (train + test) and forecast **60 days beyond the end of the
dataset**. For a feature-based ML model, forecasting forward requires generating the
lag/rolling features **iteratively** — each new day's prediction becomes an input lag for
predicting the following day, since real future lag values don't exist yet. (If the
selected best model is a classical statistical model instead, it forecasts natively via
`.forecast()`.)


In [ ]:
future_horizon = 60

if best_model_name in ml_models:
    final_model = ml_models[best_model_name].__class__(**ml_models[best_model_name].get_params())
    full_X = df_model[feature_cols]
    full_y = df_model["sales"]
    final_model.fit(full_X, full_y)

    history = df_model["sales"].copy()
    future_dates = pd.date_range(df_model.index.max() + pd.Timedelta(days=1), periods=future_horizon, freq="D")
    future_predictions = []

    for current_date in future_dates:
        row = {}
        row["year"] = current_date.year
        row["month"] = current_date.month
        row["day"] = current_date.day
        row["day_of_week"] = current_date.dayofweek
        row["week_of_year"] = int(current_date.isocalendar().week)
        row["is_weekend"] = int(current_date.dayofweek >= 5)
        row["is_month_start"] = int(current_date.is_month_start)
        row["is_month_end"] = int(current_date.is_month_end)
        row["quarter"] = current_date.quarter
        row["month_sin"] = np.sin(2 * np.pi * row["month"] / 12)
        row["month_cos"] = np.cos(2 * np.pi * row["month"] / 12)
        row["dow_sin"] = np.sin(2 * np.pi * row["day_of_week"] / 7)
        row["dow_cos"] = np.cos(2 * np.pi * row["day_of_week"] / 7)

        for lag in [1, 7, 14, 28]:
            row[f"lag_{lag}"] = history.iloc[-lag]

        shifted_history = history  # already represents "up to yesterday" relative to current_date
        for window in [7, 14, 30]:
            recent = shifted_history.iloc[-window:]
            row[f"rolling_mean_{window}"] = recent.mean()
            row[f"rolling_std_{window}"] = recent.std()
            row[f"rolling_min_{window}"] = recent.min()
            row[f"rolling_max_{window}"] = recent.max()

        row["time_index"] = len(history)

        X_future_row = pd.DataFrame([row])[feature_cols]
        y_future_pred = final_model.predict(X_future_row)[0]
        future_predictions.append(y_future_pred)

        history.loc[current_date] = y_future_pred  # feed prediction back in as the next lag source

    future_forecast = pd.Series(future_predictions, index=future_dates, name="forecast")

else:
    # Classical model: refit on full history and forecast natively
    if best_model_name == "Holt-Winters":
        final_model = ExponentialSmoothing(
            df_model["sales"], trend="add", seasonal="add", seasonal_periods=7
        ).fit()
    else:  # SARIMAX
        final_model = SARIMAX(
            df_model["sales"], order=(1, 1, 1), seasonal_order=(1, 1, 1, 7),
            enforce_stationarity=False, enforce_invertibility=False
        ).fit(disp=False)

    future_dates = pd.date_range(df_model.index.max() + pd.Timedelta(days=1), periods=future_horizon, freq="D")
    future_forecast = final_model.forecast(future_horizon)
    future_forecast.index = future_dates

print(f"Forecast generated for {future_horizon} days beyond {df_model.index.max().date()}.")
future_forecast.head(10)

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(df_model.index[-180:], df_model["sales"].iloc[-180:], label="Historical (last 180 days)", color="#4C72B0")
plt.plot(future_forecast.index, future_forecast.values, label=f"Forecast ({best_model_name})", color="#DD8452", linewidth=2)
plt.axvline(df_model.index.max(), linestyle="--", color="gray", alpha=0.7, label="Forecast Start")
plt.title(f"{future_horizon}-Day Future Sales Forecast")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Observations on Future Sales Patterns & Seasonal Behavior

*(Read this against the actual decomposition/forecast plots once the notebook has been
run — the points below describe the pattern deliberately built into the assumed dataset,
and should be re-validated against whatever a real sales export actually shows.)*

1. **Sustained upward trend** — the series was generated with steady day-over-day growth;
   in a real dataset, a similar confirmed trend (via the decomposition in Section 4.2)
   supports longer-range capacity/inventory planning, not just short-term ordering.

2. **Strong weekly seasonality** — weekend (and to a lesser extent Friday) sales are
   consistently higher. **Implication:** align staffing, ad spend pacing, and warehouse
   throughput capacity to a weekly rhythm rather than a flat daily assumption.

3. **Pronounced year-end holiday peak** (plus a smaller mid-year bump) — Section 4.3's
   monthly averages and the year-over-year comparison should show a clear Q4 spike.
   **Implication:** begin inventory build-up and marketing planning well ahead of the
   November-December peak (lead time = however long procurement/restocking actually takes),
   and treat the smaller mid-year bump as a secondary, lower-stakes planning checkpoint.

4. **Occasional promotional spikes** are real and identifiable, but short-lived and don't
   shift the underlying trend/seasonality — **implication:** flash-sale-driven spikes should
   be modeled/planned for separately (e.g. as a known campaign calendar input) rather than
   assumed to be organic demand growth when reviewing performance.

5. **Forecast uncertainty grows with horizon** — the 60-day forecast in Section 10 is most
   reliable in its first 1-2 weeks and progressively less certain further out, especially
   across the holiday season boundary where seasonal amplitude is largest.
   **Implication:** treat far-horizon forecasts as directional planning inputs, and
   re-forecast on a rolling basis (e.g. weekly) as new actuals arrive, rather than
   committing to a single static long-range number.

6. **Model choice implication:** if a feature-based ML model (Random Forest/XGBoost/
   LightGBM) won the comparison in Section 8, it likely handled the weekly+yearly
   seasonality interaction better than the purely additive classical models; if a classical
   model (Holt-Winters/SARIMAX) won, it likely benefited from directly modeling the
   autocorrelation structure with fewer parameters and less overfitting risk on a
   moderate-sized daily series. Re-confirm which family actually won once this notebook is
   executed, since that determines which approach to invest further tuning effort into.

### Recommended next steps for a production version
- Replace the synthetic data cell (Section 2) with a real historical sales export.
- Add external regressors where available (marketing spend, known promotion calendar,
  price changes, macro indicators) as exogenous features to SARIMAX or as additional ML
  features.
- Tune SARIMAX orders via `auto_arima`/grid search over AIC instead of the fixed
  `(1,1,1)x(1,1,1,7)` used here.
- Re-forecast on a rolling weekly cadence and track forecast error (MAE/MAPE) over time to
  catch model drift early.
